# LeetCode 1772 — Sort Features by Popularity bằng Python

Link đề: https://leetcode.com/problems/sort-features-by-popularity/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Sorting

> **Ghi chú (bài Premium 🔒):** đề trong notebook này được viết lại theo hiểu biết về bài và theo bản mô tả công khai của kho doocs/leetcode. Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def sortFeatures(self, features: List[str], responses: List[str]) -> List[str]:
        ...
```

## 1. Mục tiêu học

1. Tách câu thành **từ** bằng `split()` và hiểu khác biệt giữa "chứa **từ**" và "chứa **chuỗi con**".
2. Đếm **mỗi phản hồi tối đa một lần** cho mỗi tính năng bằng `set`.
3. Sắp xếp theo **nhiều khóa** (độ phổ biến giảm dần, rồi chỉ số ban đầu tăng dần) và tận dụng **tính ổn định** (stable) của `sorted`.
4. Cài thêm cách **bucket sort** theo độ phổ biến (giá trị ≤ số phản hồi).
5. Viết test có nhiều trường hợp hòa để kiểm tra luật phá hòa.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `features`, mỗi `features[i]` là **một từ** — tên một tính năng của sản phẩm. Cho mảng `responses`, mỗi `responses[i]` là một chuỗi gồm các từ cách nhau bởi dấu cách — phản hồi khảo sát.

**Độ phổ biến** của một tính năng = **số phản hồi** có chứa tính năng đó (một phản hồi chứa tính năng nhiều lần vẫn chỉ tính **một**). Sắp `features` theo độ phổ biến **không tăng**; nếu bằng nhau, giữ theo **chỉ số ban đầu** trong `features`. Trả về mảng đã sắp.

**Ví dụ 1:** `features = ["cooler","lock","touch"]`, `responses = ["i like cooler cooler","lock touch cool","locker like touch"]` → `["touch","cooler","lock"]`.
Độ phổ biến: `cooler = 1`, `lock = 1` (chú ý `"locker"` **không** phải `"lock"`), `touch = 2`.

**Ví dụ 2:** `features = ["a","aa","b","c"]`, `responses = ["a","a aa","a a a a a","b a"]` → `["a","aa","b","c"]`.

**Ràng buộc:** `1 ≤ features.length ≤ 10⁴`, `1 ≤ features[i].length ≤ 10`, `features` không trùng, chữ thường; `1 ≤ responses.length ≤ 10²`, `1 ≤ responses[i].length ≤ 10³`, chỉ chữ thường và dấu cách, không có hai dấu cách liền nhau, không có dấu cách ở đầu/cuối.

In [1]:
import random
import time
from collections import Counter
from typing import List

examples = [
    (["cooler", "lock", "touch"], ["i like cooler cooler", "lock touch cool", "locker like touch"], ["touch", "cooler", "lock"]),
    (["a", "aa", "b", "c"], ["a", "a aa", "a a a a a", "b a"], ["a", "aa", "b", "c"]),
]
for f, r, exp in examples:
    print(f, r, "->", exp)

['cooler', 'lock', 'touch'] ['i like cooler cooler', 'lock touch cool', 'locker like touch'] -> ['touch', 'cooler', 'lock']
['a', 'aa', 'b', 'c'] ['a', 'a aa', 'a a a a a', 'b a'] -> ['a', 'aa', 'b', 'c']


## 3. Kiến thức nền

### 3.1 `str.split()` — tách thành từ

In [2]:
print("locker like touch".split(), "| 'lock' in câu (chuỗi con):", "lock" in "locker like touch",
      "| 'lock' in tập từ:", "lock" in set("locker like touch".split()))

['locker', 'like', 'touch'] | 'lock' in câu (chuỗi con): True | 'lock' in tập từ: False


### 3.2 `set` — mỗi từ một lần

`set("i like cooler cooler".split())` = `{'i', 'like', 'cooler'}` ⇒ đếm mỗi phản hồi tối đa một lần cho mỗi từ.

In [3]:
print(set("i like cooler cooler".split()))

{'cooler', 'like', 'i'}


### 3.3 `Counter` và truy cập khóa không tồn tại

`Counter()[k]` trả `0` nếu `k` chưa có — tiện khi tính năng không xuất hiện trong phản hồi nào.

In [4]:
cnt = Counter(); cnt["touch"] += 2
print(cnt["touch"], cnt["không-có"])

2 0


### 3.4 `sorted` ổn định và khóa nhiều thành phần

`sorted` của Python **ổn định**: phần tử có khóa bằng nhau giữ nguyên thứ tự ban đầu. Vì vậy `sorted(features, key=lambda w: -cnt[w])` tự động phá hòa theo chỉ số ban đầu. Có thể viết tường minh `key=lambda i: (-cnt[features[i]], i)` trên chỉ số.

In [5]:
items = ["b", "a", "c"]; pop = {"a": 1, "b": 1, "c": 2}
print(sorted(items, key=lambda w: -pop[w]), sorted(range(3), key=lambda i: (-pop[items[i]], i)))
print("reverse=True vẫn ổn định:", sorted(items, key=lambda w: pop[w], reverse=True))

['c', 'b', 'a'] [2, 0, 1]
reverse=True vẫn ổn định: ['c', 'b', 'a']


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**:

| phản hồi | tập từ | tính năng có mặt |
|---|---|---|
| "i like cooler cooler" | {i, like, cooler} | cooler |
| "lock touch cool" | {lock, touch, cool} | lock, touch |
| "locker like touch" | {locker, like, touch} | touch |

⇒ `cooler: 1`, `lock: 1`, `touch: 2`. Sắp: `touch (2)`, rồi `cooler (1, chỉ số 0)`, `lock (1, chỉ số 1)`.

Hai bẫy: `"cooler cooler"` chỉ tính 1; `"locker"` và `"cool"` không phải `"lock"`, `"cooler"`.

In [6]:
f, r, _ = examples[0]
for resp in r:
    words = set(resp.split())
    print(f"{resp!r:<24} -> {sorted(w for w in f if w in words)}")

'i like cooler cooler'   -> ['cooler']
'lock touch cool'        -> ['lock', 'touch']
'locker like touch'      -> ['touch']


## 5. Cách 1 — Brute Force / baseline: với mỗi tính năng, quét mọi phản hồi

**Ý tưởng:** `pop[w] = Σ_r [w ∈ split(r)]`, rồi sắp.

In [7]:
def sort_bruteforce(features, responses):
    pop = []
    for idx, w in enumerate(features):
        c = 0
        for r in responses:
            if w in r.split():                 # list: tìm tuyến tính
                c += 1
        pop.append((-c, idx, w))
    return [w for _, _, w in sorted(pop)]

for f, r, exp in examples:
    got = sort_bruteforce(f, r); print(got, got == exp); assert got == exp

['touch', 'cooler', 'lock'] True
['a', 'aa', 'b', 'c'] True


**Độ phức tạp:** `F · R` lần tách câu + tìm trong list ⇒ `O(F · R · L)` với `L` = độ dài phản hồi: `10⁴ · 10² · 10³ = 10⁹` — **quá chậm**.

**Nút thắt:** mỗi phản hồi bị tách lại `F` lần; thay vào đó, hãy tách **một lần** và đếm cho mọi từ cùng lúc.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** đảo vòng lặp — duyệt **phản hồi** ở ngoài: tách thành **tập** từ (khử trùng trong một phản hồi), tăng bộ đếm cho mỗi từ. Sau đó độ phổ biến của tính năng `w` là `cnt[w]` (0 nếu không có).

**Quy tắc sắp:** khóa `(−cnt[w], index(w))`; hoặc chỉ `−cnt[w]` nhờ `sorted` ổn định và `features` đã theo thứ tự chỉ số.

**Tối ưu bộ nhớ (tùy chọn):** chỉ đếm từ thuộc `set(features)`.

**Bucket:** độ phổ biến nằm trong `[0, R]` (R ≤ 100) ⇒ có thể xếp vào `R + 1` "xô" theo độ phổ biến, duyệt xô từ cao xuống thấp — `O(F + R)` cho bước sắp.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `Counter` trên tập từ của mỗi phản hồi + `sorted` ổn định

In [8]:
def sort_counter(features, responses):
    cnt = Counter()
    for r in responses:
        for w in set(r.split()):
            cnt[w] += 1
    return sorted(features, key=lambda w: -cnt[w])

print([sort_counter(f, r) for f, r, _ in examples])

[['touch', 'cooler', 'lock'], ['a', 'aa', 'b', 'c']]


**Code nộp LeetCode (Cách 2):** xem mục 9. **Độ phức tạp:** đếm `O(Σ|r|)`, sắp `O(F log F)` ⇒ `O(Σ|r| + F log F)`; Space `O(số từ khác nhau)`.

### 7.2 Cách 3 — Chỉ đếm từ là tính năng, sắp trên chỉ số với khóa tường minh

In [9]:
class SolutionIndex:
    def sortFeatures(self, features: List[str], responses: List[str]) -> List[str]:
        pos = {w: i for i, w in enumerate(features)}
        pop = [0] * len(features)
        for r in responses:
            for w in set(r.split()):
                i = pos.get(w)
                if i is not None:
                    pop[i] += 1
        order = sorted(range(len(features)), key=lambda i: (-pop[i], i))
        return [features[i] for i in order]

print([SolutionIndex().sortFeatures(f, r) for f, r, _ in examples])

[['touch', 'cooler', 'lock'], ['a', 'aa', 'b', 'c']]


**Độ phức tạp Cách 3:** như Cách 2, nhưng bộ đếm chỉ có `F` phần tử. Khóa tường minh `(−pop, i)` không phụ thuộc vào tính ổn định — an toàn khi chuyển sang ngôn ngữ có sort không ổn định (C++ `std::sort`).

### 7.3 Cách 4 — Bucket sort theo độ phổ biến

In [10]:
class SolutionBucket:
    def sortFeatures(self, features: List[str], responses: List[str]) -> List[str]:
        pos = {w: i for i, w in enumerate(features)}
        pop = [0] * len(features)
        for r in responses:
            for w in set(r.split()):
                if w in pos:
                    pop[pos[w]] += 1
        buckets = [[] for _ in range(len(responses) + 1)]
        for i, w in enumerate(features):          # duyệt theo chỉ số tăng => trong mỗi xô đã đúng thứ tự
            buckets[pop[i]].append(w)
        res = []
        for b in reversed(buckets):
            res.extend(b)
        return res

print([SolutionBucket().sortFeatures(f, r) for f, r, _ in examples])

[['touch', 'cooler', 'lock'], ['a', 'aa', 'b', 'c']]


**Độ phức tạp Cách 4:** `O(Σ|r| + F + R)` — tuyến tính, không cần sắp so sánh.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 1**: bộ đếm sau mỗi phản hồi, rồi khóa sắp của từng tính năng.

In [11]:
def trace_counter(features, responses):
    cnt = Counter()
    for k, r in enumerate(responses):
        words = set(r.split())
        for w in words:
            cnt[w] += 1
        print(f"phản hồi {k}: {r!r} -> tập từ {sorted(words)} | cnt tính năng = {[(w, cnt[w]) for w in features]}")
    keys = [(w, -cnt[w], i) for i, w in enumerate(features)]
    print("khóa (tính năng, -độ phổ biến, chỉ số):", keys)
    res = sorted(features, key=lambda w: -cnt[w]); print("Kết quả:", res)
    return res

assert trace_counter(*examples[0][:2]) == examples[0][2]

phản hồi 0: 'i like cooler cooler' -> tập từ ['cooler', 'i', 'like'] | cnt tính năng = [('cooler', 1), ('lock', 0), ('touch', 0)]
phản hồi 1: 'lock touch cool' -> tập từ ['cool', 'lock', 'touch'] | cnt tính năng = [('cooler', 1), ('lock', 1), ('touch', 1)]
phản hồi 2: 'locker like touch' -> tập từ ['like', 'locker', 'touch'] | cnt tính năng = [('cooler', 1), ('lock', 1), ('touch', 2)]
khóa (tính năng, -độ phổ biến, chỉ số): [('cooler', -1, 0), ('lock', -1, 1), ('touch', -2, 2)]
Kết quả: ['touch', 'cooler', 'lock']


| tính năng | độ phổ biến | chỉ số | vị trí sau sắp |
|---|---|---|---|
| cooler | 1 | 0 | 2 |
| lock | 1 | 1 | 3 |
| touch | 2 | 2 | 1 |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — `Counter` trên tập từ + `sorted` ổn định.

In [12]:
class Solution:
    def sortFeatures(self, features: List[str], responses: List[str]) -> List[str]:
        cnt = Counter()
        for r in responses:
            for w in set(r.split()):
                cnt[w] += 1
        return sorted(features, key=lambda w: -cnt[w])

for f, r, exp in examples:
    got = Solution().sortFeatures(f, r); print(got, exp); assert got == exp

['touch', 'cooler', 'lock'] ['touch', 'cooler', 'lock']
['a', 'aa', 'b', 'c'] ['a', 'aa', 'b', 'c']


## 10. Vì sao thuật toán đúng

**Đếm đúng:** với mỗi phản hồi `r`, `set(r.split())` chứa mỗi từ của `r` đúng một lần, nên mỗi phản hồi cộng **đúng 1** cho mỗi từ có mặt ⇒ `cnt[w]` = số phản hồi chứa từ `w` = độ phổ biến (so sánh **từ nguyên vẹn**, không phải chuỗi con, vì đề định nghĩa "chứa tính năng" theo từ — Ví dụ 1 xác nhận `"locker"` không tính cho `"lock"`).

**Sắp đúng:** `sorted` sắp theo khóa `−cnt[w]` tăng dần ⇔ độ phổ biến giảm dần. Python dùng Timsort, **ổn định**: hai tính năng có khóa bằng nhau giữ thứ tự tương đối như trong `features` ⇔ chỉ số ban đầu tăng dần. ∎

**Bucket (Cách 4):** đưa tính năng vào xô theo thứ tự chỉ số tăng ⇒ trong mỗi xô thứ tự đúng; nối các xô từ độ phổ biến cao xuống thấp.

## 11. Phân tích Time / Space Complexity

Gọi `F = len(features)`, `R = len(responses)`, `T = Σ len(responses[i]) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(F · T)` | `O(L)` |
| 2. Counter + sorted | `O(T + F log F)` | `O(T)` (từ khác nhau) |
| 3. Đếm theo chỉ số | `O(T + F log F)` | `O(F)` |
| 4. Bucket | `O(T + F + R)` | `O(F + R)` |

In [13]:
rng = random.Random(1772)
alpha = "abcdefghij"
feat = list(dict.fromkeys("".join(rng.choice(alpha) for _ in range(rng.randint(1, 4))) for _ in range(12000)))[:10000]   # dict.fromkeys: khử trùng, giữ thứ tự (tái lập được)
resp = [" ".join(rng.choice(feat) if rng.random() < 0.7 else "zz" for _ in range(200)) for _ in range(100)]
for name, f in [("brute (200 tính năng)", lambda F, Rr: sort_bruteforce(F[:200], Rr)), ("Cách 2", Solution().sortFeatures),
                ("Cách 3", SolutionIndex().sortFeatures), ("Cách 4", SolutionBucket().sortFeatures)]:
    t = time.perf_counter(); out = f(feat, resp); print(f"{name:<22} {(time.perf_counter()-t)*1000:7.1f} ms | đầu: {out[:3]}")

brute (200 tính năng)    121.7 ms | đầu: ['gc', 'cihh', 'ihe']
Cách 2                     3.5 ms | đầu: ['dah', 'gdg', 'gc']


Cách 3                     3.9 ms | đầu: ['dah', 'gdg', 'gc']
Cách 4                     2.6 ms | đầu: ['dah', 'gdg', 'gc']


## 12. Bộ test

In [14]:
ALL = {"counter": sort_counter, "SolIndex": SolutionIndex().sortFeatures, "SolBucket": SolutionBucket().sortFeatures,
       "submit": Solution().sortFeatures}
def run_all(features, responses):
    ref = sort_bruteforce(features, responses)
    for k, f in ALL.items():
        assert f(features, responses) == ref, (k, features, responses)
    return ref

for f, r, exp in examples:
    assert run_all(f, r) == exp
edge = [(["x"], ["y"], ["x"]), (["b", "a"], ["a"], ["a", "b"]), (["a", "b"], ["c d"], ["a", "b"]),
        (["ab", "a"], ["ab ab ab", "abc"], ["ab", "a"]), (["z", "y", "x"], ["x y z", "x y", "x"], ["x", "y", "z"])]
for f, r, exp in edge:
    assert run_all(f, r) == exp, (f, r)
rng = random.Random(17720)
words = ["a", "b", "c", "ab", "ba", "abc", "cab", "bc"]
for _ in range(1500):
    f = rng.sample(words, rng.randint(1, len(words)))
    r = [" ".join(rng.choice(words) for _ in range(rng.randint(1, 6))) for _ in range(rng.randint(1, 5))]
    run_all(f, r)
print("Ví dụ, edge case, 1500 test ngẫu nhiên (nhiều hòa, nhiều từ là chuỗi con của nhau): OK")

Ví dụ, edge case, 1500 test ngẫu nhiên (nhiều hòa, nhiều từ là chuỗi con của nhau): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Kiểm tra chuỗi con thay vì từ (`w in response`)

In [15]:
def wrong_substring(features, responses):
    pop = {w: sum(1 for r in responses if w in r) for w in features}
    return sorted(features, key=lambda w: -pop[w])

f, r, exp = examples[0]
print("sai :", wrong_substring(f, r), "| đúng:", exp)

sai : ['lock', 'touch', 'cooler'] | đúng: ['touch', 'cooler', 'lock']


**Vì sao sai:** `"lock" in "locker like touch"` là `True` (chuỗi con), nên `lock` được tính 2. **Sửa:** so trên `r.split()`.

### Lỗi 2 — Đếm trùng trong cùng một phản hồi

In [16]:
def wrong_dup(features, responses):
    cnt = Counter()
    for r in responses:
        for w in r.split():                         # SAI: thiếu set
            cnt[w] += 1
    return sorted(features, key=lambda w: -cnt[w])

f, r = ["b", "a"], ["a a a", "b", "b"]              # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_dup(f, r), "| đúng:", Solution().sortFeatures(f, r))

sai : ['a', 'b'] | đúng: ['b', 'a']


**Vì sao sai:** `a` xuất hiện 3 lần trong **một** phản hồi nhưng chỉ được tính 1; `b` có mặt trong 2 phản hồi nên phổ biến hơn. **Sửa:** `set(r.split())`.

### Lỗi 3 — Sắp tăng rồi đảo ngược danh sách (phá luật hòa)

In [17]:
def wrong_reverse_list(features, responses):
    cnt = Counter()
    for r in responses:
        for w in set(r.split()):
            cnt[w] += 1
    return sorted(features, key=lambda w: cnt[w])[::-1]        # SAI: đảo cả thứ tự các phần tử hòa

f, r, exp = examples[0]
print("sai :", wrong_reverse_list(f, r), "| đúng:", exp)

sai : ['touch', 'lock', 'cooler'] | đúng: ['touch', 'cooler', 'lock']


**Vì sao sai:** đảo danh sách đảo luôn thứ tự giữa `cooler` và `lock`. **Sửa:** khóa `-cnt[w]`, hoặc `reverse=True` (Python giữ ổn định với `reverse=True`), hoặc khóa tường minh `(−cnt, chỉ số)`.

### Lỗi 4 — Phá hòa theo thứ tự chữ cái

In [18]:
def wrong_alpha(features, responses):
    cnt = Counter()
    for r in responses:
        for w in set(r.split()):
            cnt[w] += 1
    return sorted(features, key=lambda w: (-cnt[w], w))        # SAI: đề phá hòa theo chỉ số

f, r = ["zeta", "alpha"], ["zeta alpha"]                         # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_alpha(f, r), "| đúng:", Solution().sortFeatures(f, r))

sai : ['alpha', 'zeta'] | đúng: ['zeta', 'alpha']


**Vì sao sai:** đề yêu cầu giữ thứ tự **chỉ số ban đầu** khi hòa. **Sửa:** dùng chỉ số làm khóa phụ.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm số tài liệu chứa từ" (document frequency) ⇒ `set(doc.split())` rồi `Counter`.
- "Sắp giảm theo X, hòa thì giữ thứ tự gốc" ⇒ `sorted(..., key=lambda x: -X)` (ổn định).
- Giá trị khóa nhỏ, rời rạc ⇒ **bucket sort**.
- Luôn kiểm tra "từ" hay "chuỗi con" — một trong những bẫy đọc đề phổ biến nhất.

## 15. Biến thể của bài

1. **Đếm số lần xuất hiện** (không khử trùng) — chỉ bỏ `set`.
2. **Top-k tính năng**: `heapq.nsmallest(k, ..., key=...)` hoặc bucket.
3. **692 Top K Frequent Words**: hòa thì theo thứ tự chữ cái (ngược với bài này!).
4. **451 Sort Characters By Frequency**, **1636 Sort Array by Increasing Frequency**: sắp theo tần suất với luật phá hòa khác nhau.
5. **Tính năng gồm nhiều từ** (cụm từ): phải tìm cụm từ liên tiếp trong danh sách từ — Bài tập 5.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mỗi tính năng quét mọi phản hồi | `O(F·T)` | `O(L)` | Kiểm chứng |
| 2. Counter + sorted | Đếm một lượt, sort ổn định | `O(T + F log F)` | `O(T)` | **Bản nộp** |
| 3. Đếm theo chỉ số | Khóa `(−pop, i)` tường minh | `O(T + F log F)` | `O(F)` | Ngôn ngữ sort không ổn định |
| 4. Bucket | Xô theo độ phổ biến | `O(T + F + R)` | `O(F + R)` | Tuyến tính |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `(["a","b"], ["b","b b"])`; (b) `(["x","y","z"], ["z y", "x"])`; (c) `(["go","goo"], ["goo goo", "go"])`; (d) `(["k"], ["a b c"])`.

**Bài 2 (tính tay):** Với `features = ["p","q","r","s"]`, `responses = ["p q", "q r q", "s", "q s p"]`, tính độ phổ biến và kết quả.

**Bài 3 (điền TODO):** Hoàn thiện `popularity(features, responses)` trả list độ phổ biến theo chỉ số.

**Bài 4 (tự viết lại):** Viết `top_k(features, responses, k)` trả `k` tính năng đầu theo đúng luật của đề, dùng `heapq.nsmallest`.

**Bài 5 (mini challenge):** Tính năng có thể là **cụm nhiều từ** (ví dụ `"dark mode"`). Viết `sort_phrases(features, responses)` — một phản hồi chứa cụm nếu cụm xuất hiện như **dãy từ liên tiếp**.

In [19]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def popularity_todo(features, responses):
    pos = {w: i for i, w in enumerate(features)}
    pop = [0] * len(features)
    # TODO: với mỗi phản hồi, duyệt set(r.split()); nếu từ là tính năng thì tăng pop
    return pop

In [20]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def top_k(features, responses, k):
    # TODO: pop = popularity(...); heapq.nsmallest(k, range(F), key=lambda i: (-pop[i], i))
    pass

def sort_phrases(features, responses):
    # TODO: với mỗi phản hồi, sinh mọi cụm liên tiếp (độ dài <= số từ tối đa của tính năng) vào một set
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tách từ, khử trùng. Mức 2: đếm số phản hồi. Mức 3: (a) `["b","a"]`; (b) `["x","y","z"]`; (c) `["go","goo"]` (cùng 1, giữ chỉ số); (d) `["k"]`.

**Bài 2.** Mức 1: `p` có trong phản hồi 0, 3. Mức 2: `p = 2, q = 3, r = 1, s = 2`. Mức 3: `["q","p","s","r"]`.

**Bài 3.** Mức 1: `for r in responses`. Mức 2: `for w in set(r.split())`. Mức 3: `if w in pos: pop[pos[w]] += 1`.

**Bài 4.** Mức 1: dùng `popularity`. Mức 2: khóa `(-pop[i], i)`. Mức 3: đối chiếu với `Solution().sortFeatures(...)[:k]`.

**Bài 5.** Mức 1: `maxlen` = số từ nhiều nhất trong một tính năng. Mức 2: mọi đoạn `words[a:a+L]`, `L ≤ maxlen`, nối bằng dấu cách, cho vào `set`. Mức 3: phần còn lại như Cách 2.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
b1 = [Solution().sortFeatures(f, r) for f, r in [(["a", "b"], ["b", "b b"]), (["x", "y", "z"], ["z y", "x"]),
                                                  (["go", "goo"], ["goo goo", "go"]), (["k"], ["a b c"])]]
print("Bài 1:", b1); assert b1 == [["b", "a"], ["x", "y", "z"], ["go", "goo"], ["k"]]
f2, r2 = ["p", "q", "r", "s"], ["p q", "q r q", "s", "q s p"]
print("Bài 2:", Solution().sortFeatures(f2, r2)); assert Solution().sortFeatures(f2, r2) == ["q", "p", "s", "r"]
def popularity_todo(features, responses):
    pos = {w: i for i, w in enumerate(features)}; pop = [0] * len(features)
    for r in responses:
        for w in set(r.split()):
            if w in pos: pop[pos[w]] += 1
    return pop
print("Bài 3:", popularity_todo(f2, r2)); assert popularity_todo(f2, r2) == [2, 3, 1, 2]

Bài 1: [['b', 'a'], ['x', 'y', 'z'], ['go', 'goo'], ['k']]
Bài 2: ['q', 'p', 's', 'r']
Bài 3: [2, 3, 1, 2]


In [22]:
# Bài 4, 5
import heapq
def top_k(features, responses, k):
    pop = popularity_todo(features, responses)
    return [features[i] for i in heapq.nsmallest(k, range(len(features)), key=lambda i: (-pop[i], i))]

def sort_phrases(features, responses):
    maxlen = max(len(f.split()) for f in features)
    cnt = Counter()
    for r in responses:
        ws = r.split(); seen = set()
        for a in range(len(ws)):
            for L in range(1, maxlen + 1):
                if a + L <= len(ws):
                    seen.add(" ".join(ws[a:a + L]))
        for p in seen:
            cnt[p] += 1
    return sorted(features, key=lambda f: -cnt[f])

def phrases_brute(features, responses):
    def has(r, f):
        ws, fs = r.split(), f.split()
        return any(ws[a:a + len(fs)] == fs for a in range(len(ws) - len(fs) + 1))
    pop = [sum(has(r, f) for r in responses) for f in features]
    return [features[i] for i in sorted(range(len(features)), key=lambda i: (-pop[i], i))]

rng = random.Random(3)
for _ in range(500):
    f = rng.sample(["a", "b", "c", "ab", "bc"], rng.randint(1, 5))
    r = [" ".join(rng.choice(["a", "b", "c", "ab"]) for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(1, 4))]
    k = rng.randint(1, len(f))
    assert top_k(f, r, k) == Solution().sortFeatures(f, r)[:k]
    F = rng.sample(["dark mode", "mode", "dark", "fast app", "app"], rng.randint(1, 5))
    Rr = [" ".join(rng.choice(["dark", "mode", "fast", "app"]) for _ in range(rng.randint(1, 6))) for _ in range(rng.randint(1, 4))]
    assert sort_phrases(F, Rr) == phrases_brute(F, Rr)
print("Bài 4, 5: OK |", sort_phrases(["mode", "dark mode", "app"], ["i love dark mode", "mode dark app", "dark mode app"]))

Bài 4, 5: OK | ['mode', 'dark mode', 'app']


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "document frequency + sort ổn định":**

```python
cnt = Counter()
for r in responses:
    for w in set(r.split()):
        cnt[w] += 1
return sorted(features, key=lambda w: -cnt[w])
```

**Checklist:**
- [ ] So sánh **từ nguyên vẹn** (`split`), không phải chuỗi con?
- [ ] Mỗi phản hồi tính tối đa **một lần** cho mỗi tính năng (`set`)?
- [ ] Sắp **giảm** theo độ phổ biến, hòa thì theo **chỉ số ban đầu** (không phải chữ cái, không đảo list)?
- [ ] Tính năng không xuất hiện có độ phổ biến 0 (không lỗi `KeyError`)?
- [ ] Đã test từ là chuỗi con của từ khác, từ lặp trong một phản hồi, nhiều hòa?